# Performance Comparison of A2C and REINFORCE
## MountainCarContinuous-v0 | Team 4

In [37]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import matplotlib.pyplot as plt

In [38]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cpu")

## 1. Environment Exploration

In [39]:
env = gym.make("MountainCarContinuous-v0")
obs, _ = env.reset(seed=SEED)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
action_low = env.action_space.low[0]
action_high = env.action_space.high[0]
max_steps = env._max_episode_steps

In [40]:
print(f"State dimension  : {state_dim}")
print(f"Action dimension : {action_dim}")
print(f"Action range     : [{action_low}, {action_high}]")
print(f"Max steps        : {max_steps}")
print(f"Initial state    : {obs}")

State dimension  : 2
Action dimension : 1
Action range     : [-1.0, 1.0]
Max steps        : 999
Initial state    : [-0.4452088  0.       ]


### Random Agent Rollout

In [41]:
def random_rollout(env, seed=SEED):
    obs, _ = env.reset(seed=seed)
    total_reward = 0.0
    steps = 0
    terminated, truncated = False, False 

    while not (terminated or truncated):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        steps += 1
    
    return total_reward, steps

total_reward, steps = random_rollout(env)
print(f"Total reward : {total_reward:.2f}")
print(f"Steps taken  : {steps}")
print(f"Goal reached : {steps < 999}")

Total reward : -32.57
Steps taken  : 999
Goal reached : False


## 2. REINFORCE

In [42]:
# Policy Network
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.SELU(),
            nn.Linear(64, 64),
            nn.SELU(),
            nn.Linear(64, action_dim)
        )
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, x: torch.Tensor):
        mean = torch.tanh(self.model(x))
        std = self.log_std.exp()
        return mean, std

    def act(self, state: torch.Tensor):
        mean, std = self.forward(state)
        pd = Normal(loc=mean, scale=std)
        action = pd.sample()
        log_prob = pd.log_prob(action)
        return action.clamp(-1.0, 1.0), log_prob

### Training Function

In [43]:
def train(policy, optimizer, log_probs, rewards, gamma):
    T = len(rewards)
    returns = np.zeros(T, dtype=np.float32)
    future_return = 0.0

    for t in reversed(range(T)):
        future_return = rewards[t] + gamma * future_return
        returns[t] = future_return

    returns = torch.tensor(returns)
    log_probs = torch.stack(log_probs)

    loss = -(log_probs * returns).sum()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

### Training Loop

In [ ]:
def train_reinforce(env, policy, optimizer, gamma, num_episodes):
    episode_rewards = []

    for episode in range(num_episodes):
        state, _ = env.reset(seed=SEED + episode)
        log_probs = []
        rewards = []
        terminated, truncated = False, False

        while not (terminated or truncated):
            state_tensor = torch.FloatTensor(state)
            action, log_prob = policy.act(state_tensor)
            action_np = action.detach().numpy()

            state, reward, terminated, truncated, _ = env.step(action_np)
            log_probs.append(log_prob)
            rewards.append(reward)

        train(policy, optimizer, log_probs, rewards, gamma)
        episode_rewards.append(sum(rewards))

        if (episode + 1) % 50 == 0:
            avg = np.mean(episode_rewards[-50:])
            print(f"Episode {episode+1} | Avg Reward (last 50): {avg:.2f}")

    return episode_rewards